In [3]:
import os
import random as rnd

# Handle clean screen clearing for both Notebooks (Jupyter/VS Code) and standard Terminals
try:
    from IPython.display import clear_output

    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False

# Global list to store session match history
league_table = []


# ==========================================
# UI HELPER FUNCTIONS
# ==========================================
def clear_screen():
    """Clears the screen cleanly for both Terminal and Notebook environments."""
    if IN_NOTEBOOK:
        clear_output(wait=True)
    else:
        os.system("cls" if os.name == "nt" else "clear")


def get_hp_bar(current_hp, max_hp):
    """Generates a visual HP bar using hearts."""
    current_hp = max(0, current_hp)
    filled_hearts = "♥" * current_hp
    empty_hearts = "♡" * (max_hp - current_hp)
    return f"{filled_hearts}{empty_hearts} [{current_hp}/{max_hp}]"


def display_header(
    player_name,
    player_hp,
    boss_num,
    boss_hp,
    sym,
    max_player_hp=8,
    max_boss_hp=5,
):
    """Prints a compact, styled status header for combat."""
    player_bar = get_hp_bar(player_hp, max_player_hp)
    boss_bar = get_hp_bar(boss_hp, max_boss_hp)

    print("=" * 60)
    print(f" ENCOUNTER {boss_num}/3")
    print(
        f" {player_name.upper()}: {player_bar}  |  BOSS {boss_num}: {boss_bar}"
    )
    print(
        f" POWERS -> Rock: {sym['rock']} | Paper: {sym['paper']} | Scissors: {sym['scissors']}"
    )
    print("=" * 60)


# ==========================================
# CORE GAMEPLAY FUNCTION
# ==========================================
def run_new_game():
    """Handles setup, combat loops, upgrades, and stats for one game session."""
    clear_screen()
    print("=" * 40)
    print("          NEW GAME SETUP          ")
    print("=" * 40)

    player_name = input("Enter your hero's name: ").strip().capitalize()
    if not player_name:
        player_name = "Hero"

    # Clears screen and flushes lingering Enter key buffer from name entry
    clear_screen()

    player_hp = 8
    sym = {"rock": 1, "paper": 1, "scissors": 1}
    moves = ["rock", "paper", "scissors"]

    # Match Statistics Tracker
    stats = {
        "name": player_name,
        "rounds_played": 0,
        "remaining_hp": 0,
        "outcome": "In Progress",
        "symbol_usage": {"rock": 0, "paper": 0, "scissors": 0},
    }

    # Outer Loop: 3 Boss Encounters
    for boss_num in range(1, 4):
        boss_hp = 5
        last_round_msg = ""  # Clean start for each fight

        # Combat Loop
        while player_hp > 0 and boss_hp > 0:
            boss_symbol = rnd.choice(moves)

            # Redraw Clean UI
            clear_screen()
            display_header(player_name, player_hp, boss_num, boss_hp, sym)

            # Show outcome of previous turn only if it exists
            if last_round_msg:
                print(f"\nLAST ROUND: {last_round_msg}")

            # Input Validation Loop
            while True:
                used_symbol = (
                    input("\nMake your choice (r/p/s or rock/paper/scissors): ")
                    .lower()
                    .strip()
                )

                if used_symbol in ["r", "rock"]:
                    used_symbol = "rock"
                    break
                elif used_symbol in ["p", "paper"]:
                    used_symbol = "paper"
                    break
                elif used_symbol in ["s", "scissors"]:
                    used_symbol = "scissors"
                    break
                else:
                    # Clear screen and redraw header to clean up invalid prompt message
                    clear_screen()
                    display_header(
                        player_name, player_hp, boss_num, boss_hp, sym
                    )
                    print(
                        "\n❌ Invalid choice! Please enter 'r', 'p', or 's'."
                    )

            # Update stats
            stats["rounds_played"] += 1
            stats["symbol_usage"][used_symbol] += 1

            # Combat Resolution
            if used_symbol == boss_symbol:
                last_round_msg = (
                    f"Tie! Both chose {used_symbol.capitalize()}. No damage."
                )
            elif (
                (used_symbol == "rock" and boss_symbol == "scissors")
                or (used_symbol == "paper" and boss_symbol == "rock")
                or (used_symbol == "scissors" and boss_symbol == "paper")
            ):
                damage_dealt = sym[used_symbol]
                boss_hp -= damage_dealt
                last_round_msg = f"You win! Your {used_symbol.capitalize()} dealt {damage_dealt} damage (Boss chose {boss_symbol.capitalize()})."
            else:
                player_hp -= 1
                last_round_msg = f"Boss wins! Boss chose {boss_symbol.capitalize()} against your {used_symbol.capitalize()}. You took 1 damage."

        # Post-Fight Processing
        clear_screen()
        display_header(player_name, player_hp, boss_num, boss_hp, sym)

        if player_hp <= 0:
            print(
                f"\nGAME OVER! {player_name} was defeated in Encounter {boss_num}."
            )
            break
        else:
            print(f"\nVICTORY! You defeated Boss {boss_num}!")

            # Upgrade Phase (Boss 1 and 2 only)
            if boss_num < 3:
                print(f"\nCurrent Symbol Powers: {sym}")
                while True:
                    upgrade_choice = (
                        input(
                            "Which symbol do you want to upgrade by +1 damage? (rock/paper/scissors): "
                        )
                        .lower()
                        .strip()
                    )

                    if upgrade_choice in ["r", "rock"]:
                        upgrade_choice = "rock"
                    elif upgrade_choice in ["p", "paper"]:
                        upgrade_choice = "paper"
                    elif upgrade_choice in ["s", "scissors"]:
                        upgrade_choice = "scissors"

                    if upgrade_choice in sym:
                        sym[upgrade_choice] += 1
                        print(
                            f"\nUpgraded! {upgrade_choice.capitalize()} power is now {sym[upgrade_choice]}!"
                        )
                        input("Press Enter to continue to next encounter...")
                        break
                    else:
                        print(
                            "Invalid choice! Please select rock, paper, or scissors."
                        )

    # Record Final Results
    favorite_symbol = max(
        stats["symbol_usage"], key=stats["symbol_usage"].get
    )
    stats["remaining_hp"] = max(0, player_hp)
    stats["outcome"] = "VICTORY!" if player_hp > 0 else "DEFEATED"

    # Match Summary Screen
    print("\n" + "=" * 45)
    print(f"         FINAL MATCH SUMMARY: {stats['name']}")
    print("=" * 45)
    print(f" Outcome:          {stats['outcome']}")
    print(f" Remaining HP:     {stats['remaining_hp']} HP")
    print(f" Total Rounds:     {stats['rounds_played']}")
    print(
        f" Favorite Symbol:  {favorite_symbol.capitalize()} ({stats['symbol_usage'][favorite_symbol]} uses)"
    )
    print("=" * 45)

    league_table.append(stats)
    input("\nPress Enter to return to the Main Menu...")


# ==========================================
# LEAGUE TABLE & MENU
# ==========================================
def show_league_table():
    """Displays recorded past match statistics."""
    clear_screen()
    print("=" * 60)
    print("                       LEAGUE TABLE                       ")
    print("=" * 60)

    if not league_table:
        print("\nNo match history found. Play a game first!")
    else:
        for idx, entry in enumerate(league_table, start=1):
            fav = max(entry["symbol_usage"], key=entry["symbol_usage"].get)
            print(
                f"{idx}. {entry['name']:<10} | Status: {entry['outcome']:<9} | HP Left: {entry['remaining_hp']}/8 | Rounds: {entry['rounds_played']:<2} | Fav: {fav.capitalize()}"
            )

    print("=" * 60)
    input("\nPress Enter to return to the Main Menu...")


def main():
    """Main Menu Navigation Loop."""
    while True:
        clear_screen()
        print("==================================")
        print("      ROCK-PAPER-SCISSORS RPG     ")
        print("==================================")
        print("1. New Game")
        print("2. League Table")
        print("3. Quit")

        choice = input("\nSelect an option (1-3): ").strip()

        if choice == "1":
            run_new_game()
        elif choice == "2":
            show_league_table()
        elif choice == "3":
            clear_screen()
            print("Thanks for playing! Goodbye.")
            break
        else:
            input("Invalid choice! Press Enter to try again...")


# Run Program
if __name__ == "__main__":
    main()

Thanks for playing! Goodbye.
